# Clustered Curve Beta CDF Model

This notebook consumes the CSV produced by `custpaydetails_clustered_cumulative_curves.sql`, fits Beta CDF expected cumulative burn curves, evaluates held-out items, and compares the Beta CDF approach to a pure linear spend model.

## Input and Parameterization

| Metric | Value |
| --- | --- |
| Input CSV | clustered_data_input_adams.csv |
| Cluster curve rows | 606 |
| Items | 45 |
| Train items | 35 |
| Test items | 10 |
| Global Beta alpha | 0.979780 |
| Global Beta beta | 1.124220 |
| Global train MSE | 0.02860016 |
| Proxy label CSV | clustered_curve_proxy_labels_adams.csv |
| Proxy-labeled held-out rows | 192 |

In [ ]:
import csv
from pathlib import Path

path = Path('clustered_data_input.csv')
if not path.exists():
    path = Path('custpaydetails_clustered_cumulative_curves.csv')
if not path.exists():
    path = Path('ci_item_clustered_cumulative_curves.csv')
with path.open(newline='', encoding='utf-8-sig') as handle:
    reader = csv.DictReader(handle)
    rows = list(reader)
print(path)
print(len(rows))
print(reader.fieldnames)

## Linear Spend Baseline

The pure linear model assumes cumulative spend should equal elapsed time:

```text
expected_cumulative_pct_linear = elapsed_pct
position_delta_linear = actual_cumulative_pct - elapsed_pct
```

This is a useful baseline because it is transparent and easy to explain. It is also too rigid: many items are naturally front-loaded or back-loaded, so a linear curve can systematically flag normal burn shapes as risk.

## Held-Out Model Performance

Errors are cumulative-spend-percent errors. MAE `0.15` means an average absolute error of about 15 percentage points of final item spend.

| Model | MAE | RMSE | Median AE | P90 AE | Bias |
| --- | --- | --- | --- | --- | --- |
| Duration-bucket Beta CDF | 0.1025 | 0.1460 | 0.0666 | 0.2347 | 0.0114 |
| Linear cumulative spend | 0.1064 | 0.1484 | 0.0739 | 0.2064 | -0.0023 |
| Pooled empirical median curve | 0.1148 | 0.1550 | 0.0860 | 0.2274 | 0.0319 |
| Cluster-count-bucket Beta CDF | 0.1171 | 0.1554 | 0.1000 | 0.2139 | 0.0308 |
| Global Beta CDF | 0.1191 | 0.1537 | 0.0977 | 0.2190 | 0.0341 |

## Beta CDF vs Linear: Improvement Summary

| Comparison | Value |
| --- | --- |
| MAE improvement vs linear | 3.65% |
| RMSE improvement vs linear | 1.64% |
| P90 absolute-error improvement vs linear | -13.69% |
| Linear bias | -0.0023 |
| Duration-bucket Beta bias | 0.0114 |

## Error Comparison Plot



## Bucketed Beta CDF Parameters

### Duration Buckets

| Duration bucket | alpha | beta |
| --- | --- | --- |
| 181-365d | 1.418569 | 2.570373 |
| 366-730d | 0.789141 | 1.005997 |
| <=180d | 1.017789 | 0.936481 |
| >730d | 2.073140 | 1.868133 |

### Cluster-Count Buckets

| Cluster-count bucket | alpha | beta |
| --- | --- | --- |
| 13+ clusters | 1.257690 | 1.437277 |
| 7-12 clusters | 0.774189 | 0.826801 |

## Curve Performance Plot



## Risk Signal Framing

For active items, the same fitted curve can produce budget-overrun and time-overrun risk signals. These are not final labels without a real authorized budget and schedule; they are position signals.

```text
expected_pct = model(elapsed_pct)
position_delta = actual_cumulative_pct - expected_pct

if position_delta > threshold:
    spending too quickly / budget-overrun risk

if position_delta < -threshold:
    spending too slowly / time-overrun risk
```

For budget overrun projection once a budget is available:

```text
projected_final_spend = actual_cumulative_spend / expected_pct
projected_overrun = projected_final_spend - authorized_budget
```

## Beta vs Linear Risk Signals on Held-Out Data

The table shows how many clustered observations would be flagged by each approach at several position-delta thresholds. `LinearOnly` rows are especially important: these are cases the linear model flags but the duration-bucket Beta curve treats as normal for the observed burn shape.

| Threshold | Signal | Beta count | Linear count | Both | Linear only | Beta only |
| --- | --- | --- | --- | --- | --- | --- |
| 0.10 | spending too quickly / budget-overrun risk | 26 | 40 | 26 | 14 | 0 |
| 0.10 | spending too slowly / time-overrun risk | 40 | 43 | 34 | 9 | 6 |
| 0.15 | spending too quickly / budget-overrun risk | 19 | 27 | 19 | 8 | 0 |
| 0.15 | spending too slowly / time-overrun risk | 32 | 15 | 13 | 2 | 19 |
| 0.25 | spending too quickly / budget-overrun risk | 9 | 11 | 9 | 2 | 0 |
| 0.25 | spending too slowly / time-overrun risk | 8 | 0 | 0 | 0 | 8 |

## Proxy ROC/AUC Setup

True ROC/AUC requires true binary outcome labels. This notebook now consumes retrospective proxy labels from `clustered_curve_proxy_labels.csv`, which is generated by the separate polynomial proxy-label notebook. The Beta CDF and linear models do not create those labels; they only score against them.

These ROC curves compare how well Beta and linear position scores recover the external proxy labels. They are useful for model behavior comparison, but they are not a substitute for true budget/schedule outcome labels.

## Proxy ROC/AUC Summary

| Signal | Model | AUC | Positive labels | Negative labels |
| --- | --- | --- | --- | --- |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.9957 | 19 | 173 |
| fast_spend_proxy | Linear cumulative spend | 1.0000 | 19 | 173 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.9546 | 43 | 149 |
| slow_spend_proxy | Linear cumulative spend | 1.0000 | 43 | 149 |

## Threshold Sweep for Proxy Labels

| Signal | Model | Threshold | Flagged | TP | FP | Precision | Recall/TPR | FPR |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.05 | 50 | 19 | 31 | 0.380 | 1.000 | 0.179 |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.10 | 26 | 19 | 7 | 0.731 | 1.000 | 0.040 |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.15 | 19 | 17 | 2 | 0.895 | 0.895 | 0.012 |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.20 | 14 | 12 | 2 | 0.857 | 0.632 | 0.012 |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.25 | 9 | 9 | 0 | 1.000 | 0.474 | 0.000 |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.30 | 8 | 8 | 0 | 1.000 | 0.421 | 0.000 |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.40 | 4 | 4 | 0 | 1.000 | 0.211 | 0.000 |
| fast_spend_proxy | Linear cumulative spend | 0.05 | 47 | 19 | 28 | 0.404 | 1.000 | 0.162 |
| fast_spend_proxy | Linear cumulative spend | 0.10 | 40 | 19 | 21 | 0.475 | 1.000 | 0.121 |
| fast_spend_proxy | Linear cumulative spend | 0.15 | 27 | 19 | 8 | 0.704 | 1.000 | 0.046 |
| fast_spend_proxy | Linear cumulative spend | 0.20 | 19 | 19 | 0 | 1.000 | 1.000 | 0.000 |
| fast_spend_proxy | Linear cumulative spend | 0.25 | 11 | 11 | 0 | 1.000 | 0.579 | 0.000 |
| fast_spend_proxy | Linear cumulative spend | 0.30 | 8 | 8 | 0 | 1.000 | 0.421 | 0.000 |
| fast_spend_proxy | Linear cumulative spend | 0.40 | 7 | 7 | 0 | 1.000 | 0.368 | 0.000 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.05 | 66 | 38 | 28 | 0.576 | 0.884 | 0.188 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.10 | 40 | 34 | 6 | 0.850 | 0.791 | 0.040 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.15 | 32 | 30 | 2 | 0.938 | 0.698 | 0.013 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.20 | 21 | 20 | 1 | 0.952 | 0.465 | 0.007 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.25 | 8 | 7 | 1 | 0.875 | 0.163 | 0.007 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.30 | 2 | 2 | 0 | 1.000 | 0.047 | 0.000 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.40 | 1 | 1 | 0 | 1.000 | 0.023 | 0.000 |
| slow_spend_proxy | Linear cumulative spend | 0.05 | 71 | 43 | 28 | 0.606 | 1.000 | 0.188 |
| slow_spend_proxy | Linear cumulative spend | 0.10 | 43 | 43 | 0 | 1.000 | 1.000 | 0.000 |
| slow_spend_proxy | Linear cumulative spend | 0.15 | 15 | 15 | 0 | 1.000 | 0.349 | 0.000 |
| slow_spend_proxy | Linear cumulative spend | 0.20 | 3 | 3 | 0 | 1.000 | 0.070 | 0.000 |
| slow_spend_proxy | Linear cumulative spend | 0.25 | 0 | 0 | 0 | 0.000 | 0.000 | 0.000 |
| slow_spend_proxy | Linear cumulative spend | 0.30 | 0 | 0 | 0 | 0.000 | 0.000 | 0.000 |
| slow_spend_proxy | Linear cumulative spend | 0.40 | 0 | 0 | 0 | 0.000 | 0.000 | 0.000 |

## Proxy ROC Curves



## Residual Distribution Plot

Residual is `actual cumulative pct - expected cumulative pct`. Positive residuals indicate spending ahead of expected position; negative residuals indicate spending behind expected position.



## Recommendations

Use the duration-bucket Beta CDF as the first expected-position model and keep the linear model as a transparent benchmark. For production alerting:

- Use Beta CDF `position_delta` for primary spend-too-fast and spend-too-slow signals.
- Show the linear delta as a secondary reference because users understand it.
- Alert only when position delta exceeds a threshold and remains elevated across updates.
- For budget-overrun detection, combine position delta with authorized budget and projected final spend.
- For time-overrun detection, combine slow-spend position delta with schedule metadata; slow spending can mean delay, scope removal, or inactive work.